In [82]:
import torch 
import torch.nn.functional as F
from matplotlib import pyplot as plt 
%matplotlib inline

In [83]:
# Read the words
words = open('names.txt', 'r').read().splitlines()

# words[:8]
len(words)

32033

In [84]:
# Building the vocab of chars and mappings to and from integers
# Char to int
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

# Int to Char
itos = {i:s for s,i in stoi.items()}

print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [85]:
# # Building the dataset
# block_size = 3  # context length: how many characters do we take to predict the next one?
# X, Y = [], []  # X = inputs and Y = Next word

# for w in words:
#     context = [0] * block_size

#     for ch in w + '.':
#         ix = stoi[ch]   # Converted the chars to integers
#         X.append(context)
#         Y.append(ix)
#         # print(''.join(itos[i] for i in context), '--->', itos[ix])
#         context = context[1:] + [ix]  # Crop and append (update the context window)

# # Converting to tensors
# X = torch.tensor(X)   
# Y = torch.tensor(Y)

In [86]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [87]:
# TRAIN DEV TEST

# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      #print(''.join(itos[i] for i in context), '--->', itos[ix])
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])           # TRAIN  # 80% of the data
Xdev, Ydev = build_dataset(words[n1:n2])       # DEV    # n2 - n1 (10%)
Xte, Yte = build_dataset(words[n2:])           # TEST   # len(words) - n2  (10%)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [88]:
# Converting into embeddings
C = torch.randn((27, 2))  # Create a 27 x 2 tensor of randome numbers (embeddings)
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

### Implementing the hidden layer + internals of torch.Tensor: storage, views

In [89]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape  # cat(tensors, dimension)
# torch.cat(torch.unbind(emb, 1), 1).shape
# emb.view(32, 6).shape

In [90]:
# w1 = torch.randn((6, 100))
# b1 = torch.randn(100)

In [91]:
# h = emb.view(-1, 6) @ w1 + b1  # Hidden layer
# h.shape   # SO here (I X Y) I = training examples, Y = hidden neurons

### Implementing the output layer

In [92]:
# Weights and biases for output layer
w2 = torch.randn((100, 27))  # Because there is 27 chars (., a-z)
b2 = torch.randn(27)

In [93]:
# logits = h @ w2 + b2  # Output Lyaer
# logits.shape
# For each example (emm) network produces 27 numbers with some values and these numbers are called logits

### Implementing the negative log likelihood loss

In [94]:
# counts = logits.exp()  # To convert -ve numbers to +ve
# prob = counts / counts.sum(1, keepdims=True)  # Converting these +ve numbers into probabilities
# prob.shape

In [95]:
# loss = -prob[torch.arange(32), Y].log().mean()
# loss

### introducing F.cross_entropy

In [96]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [97]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)

W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)

W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

In [98]:
sum(p.nelement() for p in parameters)  # numbers of parameters in total

3481

In [99]:
emb = C[X]  
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2 

loss = F.cross_entropy(logits, Y)  
# loss
''' Better than performing whole expression because :
    forward and backward passes are much more efficient. 
'''

' Better than performing whole expression because :\n    forward and backward passes are much more efficient. \n'

### Implementing the training loop, overfitting one batch

In [100]:
for p in parameters: 
    p.requires_grad = True 

In [101]:
# for _ in range(100):

#     # forward pass 
#     emb = C[X]   # (32, 3, 2)
#     h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
#     logits = h @ W2 + b2   # (32, 27)
#     loss = F.cross_entropy(logits, Y)

#     # backward pass 
#     for p in parameters:
#         p.grad = None 
#     loss.backward()

#     # update 
#     for p in parameters: 
#         p.data += -0.1 * p.grad

# print(loss.item())

### finding a good initial learning rate

In [102]:
lre = torch.linspace(-3, -1, 1000)
lrs = 10**lre          # From 10^-3 to 10^0 and 1000 numbers in between

In [103]:
lr_track = []
loss_track = []

for i in range(1000):

    # minibatch construct (to get the initial learning rate)
    ix = torch.randint(0, Xtr.shape[0], (32,))

    # forward pass 
    emb = C[Xtr[ix]]   # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
    logits = h @ W2 + b2   # (32, 27)
    loss = F.cross_entropy(logits, Ytr[ix])

    # backward pass 
    for p in parameters:
        p.grad = None 
    loss.backward()

    # update 
    # lr = lrs[i]
    lr = 0.1
    for p in parameters: 
        p.data += -lr * p.grad
    # track
    lr_track.append(lr)
    loss_track.append(loss.item())

print(loss.item())

2.5474939346313477


In [104]:
# plt.plot(lr_track, loss_track)

### splitting up the dataset into train/val/test splits and why

In [110]:
'''
    train (80%) = training, optimizing parameters
    dev/validation (10%) = development of hyperparaters(size of a layer, size of embeddings) of a layer
    test (10%) = to evaluate performace of the model at the end 
'''
emb = C[Xdev]   # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
logits = h @ W2 + b2   # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

tensor(2.6614, grad_fn=<NllLossBackward0>)

In [111]:
emb = C[Xte]   # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
logits = h @ W2 + b2   # (32, 27)
loss = F.cross_entropy(logits, Yte)
loss

tensor(2.6691, grad_fn=<NllLossBackward0>)

In [112]:
# Both the above losses are same means, the data or netwrok is very small